In [ ]:
# ===== A0. Setup: analysis on CPU from the saved predictions of the main pipeline run =====
# Project: Label-free recalibration of skin cancer referral thresholds under site shift
# Author: Arshad Saeed, Bay Atlantic University
import os, sys, glob, json, importlib, numpy as np, pandas as pd
cands = [p for p in glob.glob('/kaggle/input/**/SkinCancer_Referral_Q1_2026', recursive=True) if os.path.isdir(p + '/models')]
INP = max(cands, key=lambda p: len(os.listdir(p + '/models'))); print('input:', INP)
WORK = '/kaggle/working/SkinCancer_Referral_Q1_2026'; RES = f'{WORK}/results'; os.makedirs(RES, exist_ok=True)
os.makedirs(f'{WORK}/notebooks', exist_ok=True)
sys.path.insert(0, f'{WORK}/notebooks')


In [ ]:
%%writefile /kaggle/working/SkinCancer_Referral_Q1_2026/notebooks/referral_lib.py
"""
referral_lib.py — melanoma-safe referral and label-free site recalibration.
Project: Label-free site recalibration for melanoma-safe referral (multi-center dermoscopy)
Author: Arshad Saeed, Bay Atlantic University

Policy: a case is AUTO-CLEARED (not referred) when its malignancy score s < tau, otherwise REFERRED.
Safety target: P(s < tau | malignant) <= alpha  (malignant miss rate on the deployment site).

Threshold methods (all except ORACLE use NO target labels):
  SRC      split-conformal class-conditional threshold fitted on source calibration data only
  WCP      weighted conformal (Tibshirani et al., 2019): calibration malignant scores re-weighted by
           estimated density ratio p_target(x)/p_source(x) from a cross-fitted domain classifier on embeddings.
           Per-test-case threshold; if the test point carries too much weight the case is referred
           ("certify or refuse").
  QTC      quantile threshold matching (Yilmaz & Heckel): keep the source auto-clear fraction on the target
  QTC_LS   QTC with the auto-clear fraction rescaled by the EM-estimated target benign prevalence
  ORACLE_n uses n labeled target malignant cases (few-label reference, not label-free)
"""
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

CLASSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
MAL_IDX = [0, 2, 3]   # MEL, BCC, AKIEC


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y, grid=np.linspace(0.5, 5, 46)):
    best, bt = np.inf, 1.0
    for t in grid:
        p = softmax(logits / t)
        nll = -np.log(p[np.arange(len(y)), y] + 1e-12).mean()
        if nll < best:
            best, bt = nll, t
    return bt


def mal_score(probs):
    return probs[:, MAL_IDX].sum(1)


# ---------------------------------------------------------------- thresholds
def conformal_lower(scores_mal, alpha):
    """Largest tau with P(s_new < tau) <= alpha under exchangeability (finite-sample)."""
    s = np.sort(scores_mal)
    k = int(np.floor(alpha * (len(s) + 1)))
    return -np.inf if k < 1 else s[k - 1]


def weighted_lower(scores_mal, w_cal, w_test, alpha):
    """Per-test weighted conformal lower threshold (Tibshirani et al. 2019, lower tail).
    The test point's own mass sits at -inf, so tau = -inf (refer) when that mass alone exceeds alpha."""
    order = np.argsort(scores_mal)
    s, w = scores_mal[order], w_cal[order]
    W = w.sum()
    taus = np.empty(len(w_test))
    cw = np.cumsum(w)
    for i, wt in enumerate(w_test):
        tot = W + wt
        # mass strictly below candidate s_j (exclusive) = wt + cw[j-1]
        below = (wt + np.concatenate([[0.0], cw[:-1]])) / tot
        ok = np.where(below <= alpha)[0]
        taus[i] = -np.inf if (wt / tot > alpha or len(ok) == 0) else s[ok[-1]]
    return taus


def density_ratio(emb_src, emb_tgt, n_pca=64, seed=0, clip=50.0):
    """Cross-fitted domain classifier -> w(x) = p_t(x)/p_s(x) for source AND target points.
    Returns (w_src, w_tgt, domain_auc)."""
    X = np.vstack([emb_src, emb_tgt]).astype(np.float32)
    d = np.r_[np.zeros(len(emb_src)), np.ones(len(emb_tgt))]
    X = StandardScaler().fit_transform(X)
    X = PCA(n_components=min(n_pca, X.shape[1], len(X) - 1), random_state=seed).fit_transform(X)
    p = np.zeros(len(X))
    for tr, te in StratifiedKFold(2, shuffle=True, random_state=seed).split(X, d):
        clf = LogisticRegression(C=0.1, max_iter=2000).fit(X[tr], d[tr])
        p[te] = clf.predict_proba(X[te])[:, 1]
    auc = roc_auc_score(d, p)
    p = np.clip(p, 1e-4, 1 - 1e-4)
    w = (p / (1 - p)) * (len(emb_src) / len(emb_tgt))
    w = np.clip(w, 0, clip)
    return w[:len(emb_src)], w[len(emb_src):], auc


def em_prior(probs_t, prior_s, iters=200, tol=1e-6):
    """Saerens et al. (2002) EM estimate of the target class prior (label shift), no target labels."""
    prior_s = np.clip(np.asarray(prior_s, float), 1e-6, None); prior_s = prior_s / prior_s.sum()
    pi = prior_s.copy()
    for _ in range(iters):
        q = probs_t * (pi / prior_s)
        q /= q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new
            break
        pi = new
    return pi


# ---------------------------------------------------------------- one site
def evaluate_site(src, tgt, alphas=(0.05, 0.10), oracle_ns=(20, 50), seed=0):
    """
    src: dict with  s_cal (all cal scores), y_cal_mal (0/1), emb_cal, probs_cal, prior_s
    tgt: dict with  s, y_mal (0/1, used ONLY for evaluation and ORACLE), is_mel (0/1), emb, probs, lesion (ids)
    Returns list of result rows (one per alpha x method).
    """
    rng = np.random.default_rng(seed)
    s_mal_src = src['s_cal'][src['y_cal_mal'] == 1]
    w_src, w_tgt, dom_auc = density_ratio(src['emb_cal'], tgt['emb'], seed=seed)
    w_mal = w_src[src['y_cal_mal'] == 1]
    n_eff = w_mal.sum() ** 2 / (w_mal ** 2).sum()
    pi_t = em_prior(tgt['probs'], src['prior_s'])
    mal_prev_s = src['prior_s'][MAL_IDX].sum()
    mal_prev_t_hat = pi_t[MAL_IDX].sum()
    rows = []
    for a in alphas:
        th = {}
        tau_src = conformal_lower(s_mal_src, a)
        th['SRC'] = np.full(len(tgt['s']), tau_src)
        th['WCP'] = weighted_lower(s_mal_src, w_mal, w_tgt, a)
        clear_frac_src = np.mean(src['s_cal'] < tau_src)
        th['QTC'] = np.full(len(tgt['s']), np.quantile(tgt['s'], clear_frac_src) if clear_frac_src > 0 else -np.inf)
        cf = np.clip(clear_frac_src * (1 - mal_prev_t_hat) / (1 - mal_prev_s), 0, 1)
        th['QTC_LS'] = np.full(len(tgt['s']), np.quantile(tgt['s'], cf) if cf > 0 else -np.inf)
        mal_idx = np.where(tgt['y_mal'] == 1)[0]
        for n in oracle_ns:
            if len(mal_idx) > n:
                pick = rng.choice(mal_idx, n, replace=False)
                t = conformal_lower(tgt['s'][pick], a)
                th[f'ORACLE_{n}'] = np.full(len(tgt['s']), t)
        for m, tau in th.items():
            cleared = tgt['s'] < tau
            r = summarize(cleared, tgt)
            r.update(dict(alpha=a, method=m, domain_auc=dom_auc, n_eff_mal=n_eff,
                          est_mal_prev=mal_prev_t_hat, true_mal_prev=tgt['y_mal'].mean()))
            rows.append(r)
    return rows


def summarize(cleared, tgt):
    y, mel = tgt['y_mal'] == 1, tgt['is_mel'] == 1
    return dict(
        n=len(y), n_mal=int(y.sum()), n_mel=int(mel.sum()),
        mal_miss=float(cleared[y].mean()) if y.any() else np.nan,
        mel_miss=float(cleared[mel].mean()) if mel.any() else np.nan,
        referral_rate=float((~cleared).mean()),
        auto_clear_npv=float((~y[cleared]).mean()) if cleared.any() else np.nan,
    )


def lesion_bootstrap(cleared, tgt, B=1000, seed=0):
    """95% CI for mal_miss and referral_rate, resampling lesions."""
    rng = np.random.default_rng(seed)
    les = np.asarray(tgt['lesion'])
    uniq, inv = np.unique(les, return_inverse=True)
    groups = [np.where(inv == g)[0] for g in range(len(uniq))]
    mm, rr = [], []
    for _ in range(B):
        idx = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        y = tgt['y_mal'][idx] == 1
        c = cleared[idx]
        if y.any():
            mm.append(c[y].mean())
        rr.append((~c).mean())
    return np.percentile(mm, [2.5, 97.5]), np.percentile(rr, [2.5, 97.5])


In [ ]:
# ===== A1. Load manifests + sanitize non-finite predictions (fp16 overflow) =====
import referral_lib; importlib.reload(referral_lib); from referral_lib import *
man = {n: pd.read_csv(f'{INP}/manifests/{n}.csv') for n in ['ham10000', 'bcn20000', 'milk10k', 'padufes20']}
C2I = {c: i for i, c in enumerate(CLASSES)}
runs = sorted(glob.glob(f'{INP}/models/*__split_s*/preds.npz'))
leaky = sorted(glob.glob(f'{INP}/models/*__leaky_s*/preds.npz'))
# EfficientNet-B3 predictions recomputed in fp32 (separate run) replace the fp16 ones when available
FP32 = {f.split('/')[-2]: f for f in glob.glob('/kaggle/input/**/effnet_fp32/models/*/preds.npz', recursive=True)}
runs = [FP32.get(f.split('/')[-2], f) for f in runs]; leaky = [FP32.get(f.split('/')[-2], f) for f in leaky]
print('fp32 replacements:', sorted(FP32))
print(len(runs), 'clean runs;', len(leaky), 'leaky runs')
NONFINITE = []
FORCE_REFER = np.array([20., 0, 0, 0, 0, 0, 0], np.float32)   # MEL logit high -> s(x)~1 -> always referred (conservative)
def load_preds(f):
    tag = f.split('/')[-2]; P = {}
    with np.load(f) as Z:
        for k in Z.files:
            v = Z[k].astype(np.float32)
            if v.ndim == 2 and not np.isfinite(v).all():
                rows_nan = np.isnan(v).any(1); rows_inf = np.isinf(v).any(1) & ~rows_nan
                NONFINITE.append(dict(run=tag, array=k, n_rows=len(v), rows_nan=int(rows_nan.sum()), rows_inf=int(rows_inf.sum())))
                if k.endswith('_logits'):
                    v = np.nan_to_num(v, nan=0.0, posinf=60.0, neginf=-60.0); v[rows_nan] = FORCE_REFER
                else:
                    v = np.nan_to_num(v, nan=0.0, posinf=6e4, neginf=-6e4)
            P[k] = v
    return P
_ = [load_preds(f) for f in runs + leaky]
NF = pd.DataFrame(NONFINITE); NF.to_csv(f'{RES}/nonfinite_report.csv', index=False)
print('non-finite arrays:', len(NF)); print(NF.to_string() if len(NF) else 'none')
NONFINITE.clear()


In [ ]:
# ===== ANALYSIS Cell 2: Table 2 — discrimination (in-domain vs external) + leakage audit =====
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
def bal_acc7(logits, df):
    m = df['label7'].isin(CLASSES).values
    return balanced_accuracy_score(df['label7'][m].map(C2I), logits[m].argmax(1))

rows = []
for f in runs + leaky:
    tag = f.split('/')[-2]; arch, split = tag.split('__'); seed = int(split[-2:])
    P = load_preds(f); ham = man['ham10000']
    te = (ham[split] == 'test').values
    pr = softmax(P['ham10000_logits'])
    r = dict(model=arch, split=split.split('_')[0], seed=seed,
             ham_test_balacc=bal_acc7(P['ham10000_logits'][te], ham[te]),
             ham_test_mal_auc=roc_auc_score(ham['malignant'][te], mal_score(pr[te])))
    if split.startswith('split'):
        for n in ['bcn20000', 'milk10k', 'padufes20']:
            d = man[n]; pn = softmax(P[f'{n}_logits'])
            r[f'{n}_mal_auc'] = roc_auc_score(d['malignant'], mal_score(pn))
            r[f'{n}_balacc'] = bal_acc7(P[f'{n}_logits'], d)
    rows.append(r)
T2 = pd.DataFrame(rows)
T2.to_csv(f'{RES}/table2_discrimination_raw.csv', index=False)
summ = T2.groupby(['model', 'split']).agg(['mean', 'std']).round(3)
summ.to_csv(f'{RES}/table2_discrimination_summary.csv'); print(summ.to_string())

# leakage audit: same model + seed 42, leaky vs clean
la = T2[T2.seed == 42].pivot_table(index='model', columns='split', values=['ham_test_balacc', 'ham_test_mal_auc'])
la.to_csv(f'{RES}/table3_leakage_audit.csv'); print('\nLeakage audit (seed 42):\n', la.round(3))

In [ ]:
# ===== ANALYSIS Cell 3: Main experiment — referral safety on new sites (label-free recalibration) =====
ALPHAS = (0.05, 0.10)
all_rows = []
for f in runs:
    tag = f.split('/')[-2]; arch, split = tag.split('__'); seed = int(split[-2:])
    P = load_preds(f); ham = man['ham10000']
    cal = (ham[split] == 'val').values
    T = fit_temperature(P['ham10000_logits'][cal], ham['label7'][cal].map(C2I).values)   # source-only temperature
    pc = softmax(P['ham10000_logits'][cal] / T)
    src = dict(s_cal=mal_score(pc), y_cal_mal=ham['malignant'][cal].values,
               emb_cal=P['ham10000_emb'][cal].astype(np.float32), probs_cal=pc,
               prior_s=np.bincount(ham['label7'][(ham[split] == 'train').values].map(C2I), minlength=7) / (ham[split] == 'train').sum())
    # in-domain check (HAM10000 test split) + external sites
    sites = {'ham10000_test': (ham[(ham[split] == 'test').values], P['ham10000_logits'][(ham[split] == 'test').values], P['ham10000_emb'][(ham[split] == 'test').values])}
    for n in ['bcn20000', 'milk10k', 'padufes20']:
        sites[n] = (man[n], P[f'{n}_logits'], P[f'{n}_emb'])
    mk_ = man['milk10k']
    for st_ in sorted(mk_['site'].dropna().unique()):
        m_ = (mk_['site'] == st_).values
        if m_.sum() >= 200 and mk_['malignant'][m_].sum() >= 50:
            sites[f'milk10k_{st_}'] = (mk_[m_].reset_index(drop=True), P['milk10k_logits'][m_], P['milk10k_emb'][m_])
    for site, (d, lg, em) in sites.items():
        pt = softmax(lg / T)
        tgt = dict(s=mal_score(pt), y_mal=d['malignant'].values, is_mel=(d['label7'] == 'MEL').astype(int).values,
                   emb=em.astype(np.float32), probs=pt, lesion=d['lesion_id'].values)
        for r in evaluate_site(src, tgt, alphas=ALPHAS, seed=seed):
            r.update(model=arch, seed=seed, site=site, temperature=T)
            all_rows.append(r)
    print('done', tag)
R = pd.DataFrame(all_rows)
R.to_csv(f'{RES}/table4_referral_raw.csv', index=False)
main = R.groupby(['site', 'alpha', 'method'])[['mal_miss', 'mel_miss', 'referral_rate', 'auto_clear_npv', 'domain_auc']].agg(['mean', 'std'])
main.round(3).to_csv(f'{RES}/table4_referral_summary.csv')
print(main.round(3).to_string())

In [ ]:
# ===== ANALYSIS Cell 4: Safety violations — how often each method breaks the promised miss rate =====
R['violation'] = R['mal_miss'] > R['alpha']
viol = R.groupby(['alpha', 'method', 'site'])['violation'].mean().unstack('site').round(2)
viol.to_csv(f'{RES}/table5_violation_rate.csv'); print(viol)
by_model = R[R.alpha == 0.05].pivot_table(index=['model'], columns=['site', 'method'], values='mal_miss', aggfunc='mean').round(3)
by_model.to_csv(f'{RES}/table6_missrate_by_model.csv')
pd.set_option('display.width', 250)
for a in (0.05, 0.10):
    ext = R[(R.alpha == a) & R.site.isin(['bcn20000', 'milk10k', 'padufes20'])]
    print(f'\nPer-model (alpha={a}), mean over seeds and external sites:')
    print(ext.pivot_table(index='model', columns='method', values=['mal_miss', 'referral_rate', 'violation'], aggfunc='mean').round(3).to_string())


In [ ]:
# ===== FIG 0. Setup: read outputs of the main pipeline run =====
# Project: Label-free site recalibration for melanoma-safe referral
# Author: Arshad Saeed, Bay Atlantic University
import os, sys, glob, json, importlib, numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT = '/kaggle/working/figures'; os.makedirs(OUT, exist_ok=True)
import referral_lib; importlib.reload(referral_lib); from referral_lib import *
man = {n: pd.read_csv(f'{INP}/manifests/{n}.csv') for n in ['ham10000', 'bcn20000', 'milk10k', 'padufes20']}
C2I = {c: i for i, c in enumerate(CLASSES)}
R = pd.read_csv(f'{RES}/table4_referral_raw.csv')
T2 = pd.read_csv(f'{RES}/table2_discrimination_raw.csv')
print('input:', INP, '| referral rows:', len(R), '| sites:', sorted(R.site.unique()))
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150})
METHOD_ORDER = ['SRC', 'QTC', 'QTC_LS', 'WCP', 'ORACLE_20', 'ORACLE_50']
METHOD_LABEL = {'SRC': 'Source-only', 'QTC': 'QTC', 'QTC_LS': 'QTC + label shift', 'WCP': 'Weighted conformal',
                'ORACLE_20': 'Oracle (20 labels)', 'ORACLE_50': 'Oracle (50 labels)'}
SITE_LABEL = lambda s: s.replace('ham10000_test', 'HAM10000 (internal)').replace('bcn20000', 'BCN20000').replace('padufes20', 'PAD-UFES-20 (phone)').replace('milk10k_', 'MILK10k: ').replace('milk10k', 'MILK10k (all)')

In [ ]:
# ===== FIG 1. Leakage audit: image-level vs lesion-level split (seed 42) =====
la = T2[T2.seed == 42].pivot_table(index='model', columns='split', values='ham_test_balacc').dropna()
fig, ax = plt.subplots(figsize=(5, 2.6))
x = np.arange(len(la))
ax.bar(x - 0.18, la['leaky'], 0.36, label='Image-level split (leaky)', color='#bbbbbb')
ax.bar(x + 0.18, la['split'], 0.36, label='Lesion-level split (clean)', color='#2b6cb0')
for i, (a, b) in enumerate(zip(la['leaky'], la['split'])):
    ax.text(i, max(a, b) + 0.01, f'+{(a - b) * 100:.1f} pts', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(la.index, rotation=15); ax.set_ylabel('Balanced accuracy (HAM10000 test)')
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=8, loc='lower right')
ax.set_title('Image-level splitting inflates HAM10000 results')
fig.tight_layout(); fig.savefig(f'{OUT}/fig_leakage_audit.png', dpi=300); fig.savefig(f'{OUT}/fig_leakage_audit.pdf')
la.assign(inflation_pts=(la['leaky'] - la['split']) * 100).round(3).to_csv(f'{OUT}/table_leakage_audit.csv'); print(la.round(3))

In [ ]:
# ===== FIG 2. Main result: malignant miss rate per site and method (alpha = 0.05) =====
def site_order(df):
    s = sorted(df.site.unique(), key=lambda v: (v != 'ham10000_test', v.startswith('milk10k_'), v))
    return s
for a in sorted(R.alpha.unique()):
    d = R[R.alpha == a]
    sites = site_order(d); meths = [m for m in METHOD_ORDER if m in d.method.unique()]
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(sites) + 1.2))
    for j, m in enumerate(meths):
        g = d[d.method == m].groupby('site')['mal_miss']
        y = np.arange(len(sites)) + (j - (len(meths) - 1) / 2) * 0.12
        med = [g.median().get(s, np.nan) for s in sites]; lo = [g.min().get(s, np.nan) for s in sites]; hi = [g.max().get(s, np.nan) for s in sites]
        ax.errorbar(med, y, xerr=[np.array(med) - lo, np.array(hi) - med], fmt='o', ms=3.5, lw=1, capsize=0,
                    label=METHOD_LABEL[m], color=plt.cm.tab10(j))
    ax.axvline(a, ls='--', color='k', lw=0.8)
    ax.text(a, 1.0, f' promised ≤ {a:.0%}', fontsize=8, va='bottom', transform=ax.get_xaxis_transform())
    ax.set_yticks(np.arange(len(sites))); ax.set_yticklabels([SITE_LABEL(s) for s in sites]); ax.invert_yaxis()
    ax.set_xlabel('Share of malignant cases auto-cleared (median and range over models × seeds)')
    ax.legend(frameon=False, fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1.0))
    ax.set_title(f'Does the referral rule keep its promise on new sites? (α = {a:.2f})', pad=14)
    fig.tight_layout(); fig.savefig(f'{OUT}/fig_missrate_alpha{int(a*100):02d}.png', dpi=300, bbox_inches='tight'); fig.savefig(f'{OUT}/fig_missrate_alpha{int(a*100):02d}.pdf', bbox_inches='tight')

In [ ]:
# ===== FIG 3. Safety vs workload: miss rate vs referral rate (alpha = 0.05) =====
d = R[(R.alpha == 0.05) & (R.site != 'ham10000_test')]
fig, ax = plt.subplots(figsize=(5, 3.4))
for j, m in enumerate([m for m in METHOD_ORDER if m in d.method.unique()]):
    g = d[d.method == m].groupby('site')[['mal_miss', 'referral_rate']].mean()
    ax.scatter(g['referral_rate'], g['mal_miss'], s=18, label=METHOD_LABEL[m], color=plt.cm.tab10(j))
ax.axhline(0.05, ls='--', color='k', lw=0.8)
ax.set_xlabel('Referral rate (share of cases sent to dermatologist)'); ax.set_ylabel('Malignant miss rate')
ax.legend(frameon=False, fontsize=7); ax.set_title('Safety vs workload per site (mean over models)')
fig.tight_layout(); fig.savefig(f'{OUT}/fig_safety_vs_workload.png', dpi=300); fig.savefig(f'{OUT}/fig_safety_vs_workload.pdf')

In [ ]:
# ===== FIG 4. Shift size vs safety breach (source-only vs weighted conformal) =====
d = R[(R.alpha == 0.05) & R.method.isin(['SRC', 'WCP'])]
fig, ax = plt.subplots(figsize=(5, 3.2))
for j, m in enumerate(['SRC', 'WCP']):
    g = d[d.method == m]
    ax.scatter(g['domain_auc'], g['mal_miss'], s=8, alpha=0.5, label=METHOD_LABEL[m], color=plt.cm.tab10([0, 3][j]))
ax.axhline(0.05, ls='--', color='k', lw=0.8)
ax.set_xlabel('Domain-classifier AUROC (how different the site is)'); ax.set_ylabel('Malignant miss rate')
ax.legend(frameon=False, fontsize=8); ax.set_title('Larger shift, larger breach for the source-only rule')
fig.tight_layout(); fig.savefig(f'{OUT}/fig_shift_vs_breach.png', dpi=300); fig.savefig(f'{OUT}/fig_shift_vs_breach.pdf')
print(d.groupby('method')[['domain_auc', 'mal_miss']].corr().round(3))

In [ ]:
# ===== TABLE. Per-image decisions for one model -> lesion-bootstrap CIs, skin tone and artifact audit =====
def decisions(f, alpha=0.05, seed=42):
    """Recompute per-image auto-clear decisions for every site and method for one run."""
    tag = f.split('/')[-2]; arch, split = tag.split('__')
    P = load_preds(f); ham = man['ham10000']
    cal = (ham[split] == 'val').values
    T = fit_temperature(P['ham10000_logits'][cal], ham['label7'][cal].map(C2I).values)
    pc = softmax(P['ham10000_logits'][cal] / T); s_cal = mal_score(pc); y_cal = ham['malignant'][cal].values
    s_mal = s_cal[y_cal == 1]
    out = {}
    sites = {'bcn20000': man['bcn20000'], 'milk10k': man['milk10k'], 'padufes20': man['padufes20']}
    for n, d_ in sites.items():
        pt = softmax(P[f'{n}_logits'] / T); s = mal_score(pt)
        w_src, w_tgt, _ = density_ratio(P['ham10000_emb'][cal].astype(np.float32), P[f'{n}_emb'].astype(np.float32), seed=seed)
        tau = {'SRC': np.full(len(s), conformal_lower(s_mal, alpha)),
               'WCP': weighted_lower(s_mal, w_src[y_cal == 1], w_tgt, alpha)}
        out[n] = pd.DataFrame({'image_id': d_['image_id'], 'lesion_id': d_['lesion_id'], 'malignant': d_['malignant'],
                               'label7': d_['label7'], 's': s, **{f'clear_{k}': s < v for k, v in tau.items()}})
    return arch, out

runs = [FP32.get(f.split('/')[-2], f) for f in sorted(glob.glob(f'{INP}/models/*__split_s42/preds.npz'))]
best = T2[(T2.split == 'split') & (T2.seed == 42)].set_index('model')[[c for c in T2.columns if c.endswith('_mal_auc') and not c.startswith('ham')]].mean(axis=1).idxmax()
f_best = [f for f in runs if f.split('/')[-2].startswith(best + '__')][0]
arch, D = decisions(f_best)
print('model used for per-image analysis:', arch)

rows = []
for n, df in D.items():
    for m in ['SRC', 'WCP']:
        tgt = dict(y_mal=df['malignant'].values, lesion=df['lesion_id'].values)
        c = df[f'clear_{m}'].values
        (mlo, mhi), (rlo, rhi) = lesion_bootstrap(c, tgt, B=1000)
        rows.append(dict(site=n, method=m, mal_miss=c[df.malignant == 1].mean(), miss_ci=f'{mlo:.3f}–{mhi:.3f}',
                         referral=(~c).mean(), referral_ci=f'{rlo:.3f}–{rhi:.3f}'))
CI = pd.DataFrame(rows); CI.round(3).to_csv(f'{OUT}/table_bootstrap_ci_{arch}.csv', index=False); print(CI.round(3).to_string(index=False))

In [ ]:
# ===== AUDIT 1. MILK10k skin tone (descriptive only: very few dark-skin images) =====
mk = man['milk10k'][['image_id', 'skin_tone_class']].merge(D['milk10k'], on='image_id')
mk['tone_group'] = pd.cut(mk['skin_tone_class'], [-1, 2, 3, 5], labels=['0–2 (darker)', '3 (medium)', '4–5 (lighter)'])
st = (mk[mk.malignant == 1].groupby('tone_group', observed=True)
      .agg(n_malignant=('malignant', 'size'), miss_SRC=('clear_SRC', 'mean'), miss_WCP=('clear_WCP', 'mean')))
st.round(3).to_csv(f'{OUT}/table_skin_tone_audit_{arch}.csv'); print(st.round(3))

In [ ]:
# ===== AUDIT 2. Artifacts (MONET concept scores) among missed malignant cases =====
monet = [c for c in man['milk10k'].columns if c.startswith('MONET_')]
mm = man['milk10k'][['image_id'] + monet].merge(D['milk10k'], on='image_id')
mal = mm[mm.malignant == 1]
art = pd.DataFrame({c: [mal.loc[mal.clear_SRC, c].mean(), mal.loc[~mal.clear_SRC, c].mean()] for c in monet},
                   index=['missed (auto-cleared)', 'caught (referred)']).T
art['difference'] = art.iloc[:, 0] - art.iloc[:, 1]
art.sort_values('difference', ascending=False).round(3).to_csv(f'{OUT}/table_artifact_audit_{arch}.csv'); print(art.sort_values('difference', ascending=False).round(3))
print('\nSaved figures and tables to', OUT, ':', sorted(os.listdir(OUT)))

In [ ]:
# ===== SENSITIVITY. PAD-UFES-20 with actinic keratosis (ACK) counted as benign =====
pad = man['padufes20'][['image_id', 'diagnostic']].merge(D['padufes20'], on='image_id')
rows = []
for defn, ymal in [('ACK malignant (main)', pad['malignant'] == 1),
                   ('ACK benign (sensitivity)', pad['diagnostic'].isin(['BCC', 'SCC', 'MEL']))]:
    for m in ['SRC', 'WCP']:
        c = pad[f'clear_{m}'].values
        rows.append(dict(definition=defn, method=m, n_malignant=int(ymal.sum()),
                         mal_miss=float(c[ymal.values].mean()), referral=float((~c).mean())))
SENS = pd.DataFrame(rows); SENS.round(3).to_csv(f'{OUT}/table_sensitivity_pad_ack_{arch}.csv', index=False); print(SENS.round(3).to_string(index=False))

In [ ]:
# ===== Z. Manuscript-ready tables + key numbers =====
import sys
sys.argv = ["make_results.py", RES, f"{RES}/results_for_manuscript.md"]

import json

OUT = sys.argv[2]
NAME = {'resnet50': 'ResNet-50', 'mobilenetv2': 'MobileNetV2', 'xception': 'Xception',
        'efficientnet_b3': 'EfficientNet-B3', 'convnext_tiny': 'ConvNeXt-T', 'dinov2_vitb14': 'DINOv2 (probe)'}
SITE = {'ham10000_test': 'HAM10000 (internal test)', 'bcn20000': 'BCN20000', 'milk10k': 'MILK10k (all)',
        'padufes20': 'PAD-UFES-20'}
METH = ['SRC', 'WCP', 'QTC', 'QTC_LS', 'ORACLE_20', 'ORACLE_50']
MLAB = {'SRC': 'Source-only (SRC)', 'WCP': 'Weighted conformal (WCP)', 'QTC': 'Quantile transfer (QTC)',
        'QTC_LS': 'QTC + label shift', 'ORACLE_20': 'Oracle, 20 labels', 'ORACLE_50': 'Oracle, 50 labels'}

def nm(m):
    for k, v in NAME.items():
        if m.startswith(k): return v
    return m
def site(s): return SITE.get(s, s.replace('milk10k_', 'MILK10k – '))
def ms(x): return f'{np.nanmean(x):.3f} ± {np.nanstd(x, ddof=1) if len(x) > 1 else 0:.3f}'
def pct(x): return f'{100 * x:.1f}%'
def md(df):
    cols = list(df.columns)
    return '\n'.join(['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)] +
                     ['| ' + ' | '.join(str(v) for v in r) + ' |' for r in df.values])

T2 = pd.read_csv(f'{RES}/table2_discrimination_raw.csv')
R = pd.read_csv(f'{RES}/table4_referral_raw.csv')
key, out = {}, []

# ---- Table 2: discrimination (lesion-level split, mean ± sd over seeds) ----
cl = T2[T2.split == 'split']
rows = []
for m, g in cl.groupby('model'):
    r = {'Model': nm(m), 'HAM bal. acc.': ms(g.ham_test_balacc), 'HAM malignant AUROC': ms(g.ham_test_mal_auc)}
    for s in ['bcn20000', 'milk10k', 'padufes20']:
        if f'{s}_mal_auc' in g: r[f'{SITE[s]} AUROC'] = ms(g[f'{s}_mal_auc'])
    rows.append(r)
t2 = pd.DataFrame(rows); out += ['## Table 2. Discrimination (lesion-level split, mean ± SD over seeds)', md(t2), '']

# ---- Table 3: leakage audit ----
lk = T2[T2.seed == 42].pivot_table(index='model', columns='split', values=['ham_test_balacc', 'ham_test_mal_auc'])
if ('ham_test_balacc', 'leaky') in lk.columns:
    lk = lk.dropna()
    t3 = pd.DataFrame({'Model': [nm(m) for m in lk.index],
                       'Bal. acc. image split': lk[('ham_test_balacc', 'leaky')].round(3).values,
                       'Bal. acc. lesion split': lk[('ham_test_balacc', 'split')].round(3).values,
                       'Inflation (pts)': ((lk[('ham_test_balacc', 'leaky')] - lk[('ham_test_balacc', 'split')]) * 100).round(1).values,
                       'AUROC image split': lk[('ham_test_mal_auc', 'leaky')].round(3).values,
                       'AUROC lesion split': lk[('ham_test_mal_auc', 'split')].round(3).values})
    out += ['## Table 3. Leakage audit (seed 42)', md(t3), '']
    infl = t3['Inflation (pts)']
    key['leak_inflation_range_pts'] = f'{infl.min():.1f}–{infl.max():.1f}'
    key['leak_inflation_mean_pts'] = round(float(infl.mean()), 1)

# ---- Table 4: referral safety ----
R['violation'] = R.mal_miss > R.alpha
for a in sorted(R.alpha.unique()):
    rows = []
    for s in [x for x in ['ham10000_test', 'bcn20000', 'milk10k', 'padufes20'] if x in R.site.unique()] + \
             sorted(x for x in R.site.unique() if x.startswith('milk10k_')):
        for m in METH:
            g = R[(R.alpha == a) & (R.site == s) & (R.method == m)]
            if g.empty: continue
            rows.append({'Site': site(s), 'Method': MLAB[m], 'Malignant miss': ms(g.mal_miss),
                         'Melanoma miss': ms(g.mel_miss), 'Referral rate': ms(g.referral_rate),
                         'NPV auto-cleared': ms(g.auto_clear_npv), 'Runs breaking promise': pct(g.violation.mean()),
                         'Domain AUC': f'{g.domain_auc.mean():.2f}'})
    out += [f'## Table 4. Referral safety at α = {a:.2f} (mean ± SD over 6 models × 3 seeds)', md(pd.DataFrame(rows)), '']


# ---- Table 5: paired statistical tests (same model, seed, site, alpha) ----
from itertools import combinations
from scipy.stats import wilcoxon, binomtest
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o):
        run = max(run, min(1, (m - r) * p[i])); adj[i] = run
    return adj
EXT = ['bcn20000', 'milk10k', 'padufes20']
LF = ['SRC', 'WCP', 'QTC', 'QTC_LS']
rows = []
for a in sorted(R.alpha.unique()):
    sub = R[(R.alpha == a) & R.site.isin(EXT) & R.method.isin(LF)]
    P = sub.pivot_table(index=['model', 'seed', 'site'], columns='method', values=['mal_miss', 'referral_rate', 'violation'])
    for m1, m2 in combinations(LF, 2):
        r = {'alpha': a, 'Comparison': f'{MLAB[m1]} vs {MLAB[m2]}', 'n pairs': len(P)}
        for v in ['mal_miss', 'referral_rate']:
            d = (P[(v, m1)] - P[(v, m2)]).dropna()
            r[f'Δ {v} (median)'] = round(float(d.median()), 3)
            r[f'p {v}'] = float(wilcoxon(d, zero_method='zsplit').pvalue) if (d != 0).any() else 1.0
        b = (P[('violation', m1)] > 0.5) & (P[('violation', m2)] < 0.5)
        c = (P[('violation', m1)] < 0.5) & (P[('violation', m2)] > 0.5)
        r['breaks only first / only second'] = f'{int(b.sum())} / {int(c.sum())}'
        r['p violation (McNemar)'] = float(binomtest(int(b.sum()), int(b.sum() + c.sum()), 0.5).pvalue) if (b.sum() + c.sum()) else 1.0
        rows.append(r)
T5 = pd.DataFrame(rows)
for col in [c for c in T5.columns if c.startswith('p ')]:
    T5[col + ' (Holm)'] = T5.groupby('alpha')[col].transform(lambda x: holm(x.values)).map(lambda v: f'{v:.3g}')
    T5 = T5.drop(columns=col)
out += ['## Table 5. Paired comparisons across external sites (Wilcoxon / exact McNemar, Holm-adjusted)', md(T5), '']

# ---- key numbers for abstract / highlights / cover letter ----
ext = R[(R.alpha == 0.05) & (R.site.isin(['bcn20000', 'milk10k', 'padufes20']))]
for m in ['SRC', 'WCP', 'QTC', 'QTC_LS']:
    g = ext[ext.method == m]
    key[f'{m}_ext_mal_miss_mean'] = round(float(g.mal_miss.mean()), 3)
    key[f'{m}_ext_mal_miss_max_site'] = round(float(g.groupby('site').mal_miss.mean().max()), 3)
    key[f'{m}_ext_violation_rate'] = round(float(g.violation.mean()), 3)
    key[f'{m}_ext_referral_mean'] = round(float(g.referral_rate.mean()), 3)
    key[f'{m}_ext_mel_miss_mean'] = round(float(g.mel_miss.mean()), 3)
per_site = ext.groupby(['site', 'method']).mal_miss.mean().unstack().round(3)
out += ['## Key: mean malignant miss at α = 0.05 by site', md(per_site.reset_index()), '',
        '## Key numbers', '```json', json.dumps(key, indent=1), '```']
key['n_runs'] = int(R.groupby(['model', 'seed']).ngroups)
open(OUT, 'w').write('\n'.join(out))
json.dump(key, open(OUT.replace('.md', '_key.json'), 'w'), indent=1)
print('\n'.join(out))


In [ ]:
# ===== Z2. Zip everything for download =====
import shutil
shutil.make_archive('/kaggle/working/skin_referral_results', 'zip', '/kaggle/working', '.')
print(sorted(os.listdir(RES))); print(sorted(os.listdir('/kaggle/working/figures')))